In [1]:
import os
import time
import numpy as np
from tqdm import tqdm

import pybullet as p
import pybullet_data
from surrol.utils.pybullet_utils import (
    step,
    get_joints,
    get_link_name,
    reset_camera,
    get_link_pose
)
from surrol.robots.psm import Psm

p.connect(p.GUI)
# p.connect(p.DIRECT)
p.setGravity(0, 0, -9.81)

POSE_TRAY = ((0.55, 0, 0.6751), (0, 0, 0))
SCALING = 5.
workspace_limits = np.array([(2.5, 3), (-0.25, 0.25), (3.476, 3.776)])

tray_id = p.loadURDF('/Users/kantaphat/Research/DEX/SurRoL/surrol/assets/tray/tray_pad.urdf',
                    np.array(POSE_TRAY[0]) * SCALING,
                    p.getQuaternionFromEuler(POSE_TRAY[1]),
                    globalScaling=SCALING)

reset_camera(yaw=90, pitch=-40, dist=1, target=np.array(POSE_TRAY[0]) * SCALING)

POSE_PSM1 = ((0.05, 0.24, 0.8524), (0, 0, -(90 + 20) / 180 * np.pi))
psm = Psm(POSE_PSM1[0], 
          p.getQuaternionFromEuler(POSE_PSM1[1]), 
          scaling=SCALING)

pybullet build time: Feb 15 2025 12:06:37


Version = 4.1 Metal - 89.4
Vendor = Apple
Renderer = Apple M1
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started


In [2]:
pos = (workspace_limits[0][0],
       workspace_limits[1][1],
      (workspace_limits[2][1] + workspace_limits[2][0]) / 2)
orn = (0.5, 0.5, -0.5, -0.5)
joint_positions = psm.inverse_kinematics((pos, orn), psm.EEF_LINK_INDEX)
psm.reset_joint(joint_positions)

array([ 0.19266954, -0.00727711,  0.13608575, -0.3429539 , -0.05868378,
       -0.18376323])

In [4]:
sphere_radius = 0.1
sphere_pos = [
    workspace_limits[0].mean(), 
    workspace_limits[1].mean(), 
    workspace_limits[2][0] + sphere_radius - 0.03
]

sphere_id = p.createMultiBody(
    baseMass=0,
    baseVisualShapeIndex=p.createVisualShape(
        p.GEOM_SPHERE, 
        radius=sphere_radius, 
        rgbaColor=[0, 1, 0, 0.3]
    ),
    basePosition=sphere_pos,
    baseOrientation=p.getQuaternionFromEuler([0, 0, 0])
)

In [ ]:
p.removeBody(sphere_id)

In [5]:
reset_camera(yaw=90, pitch=-30, dist=0.6, target=np.array(POSE_TRAY[0]) * SCALING)

In [3]:
def draw_workspace_box(limits, color=[0, 0, 1], line_width=1, lifetime=0):
    """
    Draw a bounding box for workspace limits in PyBullet
    
    Args:
        limits: ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        color: [r, g, b] line color
        line_width: width of the lines
        lifetime: 0 for permanent, >0 for temporary in seconds
    """
    x_min, x_max = limits[0]
    y_min, y_max = limits[1] 
    z_min, z_max = limits[2]
    
    # Define the 8 corners of the box
    corners = [
        [x_min, y_min, z_min], [x_max, y_min, z_min],
        [x_min, y_max, z_min], [x_max, y_max, z_min],
        [x_min, y_min, z_max], [x_max, y_min, z_max],
        [x_min, y_max, z_max], [x_max, y_max, z_max]
    ]
    
    # Define the 12 edges of the box
    edges = [
        # Bottom face (z_min)
        (0, 1), (0, 2), (1, 3), (2, 3),
        # Top face (z_max)
        (4, 5), (4, 6), (5, 7), (6, 7),
        # Vertical edges
        (0, 4), (1, 5), (2, 6), (3, 7)
    ]
    
    # Draw all edges
    line_ids = []
    for edge in edges:
        start_pos = corners[edge[0]]
        end_pos = corners[edge[1]]
        line_id = p.addUserDebugLine(start_pos, end_pos, color, line_width, lifetime)
        line_ids.append(line_id)
    
    return line_ids

# Remove the box later
def remove_workspace_box(line_ids):
    """Remove previously drawn workspace box"""
    for line_id in line_ids:
        p.removeUserDebugItem(line_id)

In [7]:
workspace_id = draw_workspace_box(workspace_limits, color=[0, 0, 1])

In [ ]:
remove_workspace_box(workspace_id)

In [4]:
original_needle_ranges = np.array((
    (workspace_limits[0].mean() - 0.05, workspace_limits[0].mean() + 0.05),
    (workspace_limits[1].mean() - 0.05, workspace_limits[1].mean() + 0.05),
    (workspace_limits[2][0] + 0.01, workspace_limits[2][0] + 0.009)
))

original_needle_ranges_id = draw_workspace_box(original_needle_ranges, color=[0, 1, 0])

In [5]:
# needle_ranges = np.array((
#     (workspace_limits[0].mean() - 0.05, workspace_limits[0].mean() + 0.05),
#     (workspace_limits[1][0], workspace_limits[1][0] + 0.08),
#     (workspace_limits[2][0] + 0.01, workspace_limits[2][0] + 0.009)
# ))

needle_ranges = np.array((
    (workspace_limits[0].mean(), workspace_limits[0].mean() + 0.1),
    (workspace_limits[1][0], workspace_limits[1][0] - 0.001),
    (workspace_limits[2][0] + 0.01, workspace_limits[2][0] + 0.009)
))

needle_ranges_id = draw_workspace_box(needle_ranges, color=[1, 0, 0])

In [21]:
remove_workspace_box(needle_ranges_id)
remove_workspace_box(original_needle_ranges_id)

In [6]:
pick_up_ranges = np.array((
    (workspace_limits[0].mean() - 0.1, workspace_limits[0].mean() + 0.1),
    (workspace_limits[1][0], workspace_limits[1][0] + 0.1),
    (workspace_limits[2][0] + 0.01, workspace_limits[2][0] + 0.009)
))

pick_up_ranges_id = draw_workspace_box(pick_up_ranges, color=[1, 1, 0])

In [20]:
remove_workspace_box(pick_up_ranges_id)

In [7]:
original_goal_ranges = np.array((
    (workspace_limits[0].mean() + 0.01 * -2.576 * SCALING, workspace_limits[0].mean() + 0.01 * 2.576 * SCALING),
    (workspace_limits[1].mean() + 0.01 * -2.576 * SCALING, workspace_limits[1].mean() + 0.01 * 2.576 * SCALING),
    (workspace_limits[2][1] - 0.04 * SCALING, workspace_limits[2][1] - 0.039 * SCALING)
))

goal_ranges = np.array((
    (workspace_limits[0].mean() + 0.01 * -2.576 * SCALING, workspace_limits[0].mean() + 0.01 * 2.576 * SCALING),
    (workspace_limits[1].mean() + 0.01 * SCALING * 2.576, workspace_limits[1][1]),
    (workspace_limits[2][1] - 0.04 * SCALING, workspace_limits[2][1] - 0.039 * SCALING) # Drawing purpose
))

goal_id = draw_workspace_box(goal_ranges, color=[0, 0, 0])
original_goal_id = draw_workspace_box(original_goal_ranges, color=[0, 1, 1])


In [ ]:
remove_workspace_box(goal_id)
remove_workspace_box(original_goal_id)

In [8]:
needle_radius = 0.1

for _ in range(1000):
    yaw = (np.random.rand() - 0.5) * np.pi
    needle_pos = (
        np.random.uniform(needle_ranges[0][0], needle_ranges[0][1]),
        workspace_limits[1][0],
        workspace_limits[2][0] + 0.01
    )

    pick_up_pos =(
        needle_pos[0] - needle_radius * np.cos(yaw),
        needle_pos[1] - needle_radius * np.sin(yaw),
        needle_pos[2]
    )

    x, y, _ = pick_up_pos
    x_min, x_max = pick_up_ranges[0][0], pick_up_ranges[0][1]
    y_min, y_max = pick_up_ranges[1][0], pick_up_ranges[1][1]
    
    if (x_min <= x <= x_max and y_min <= y <= y_max):
        print(0.25 + pick_up_pos[1], yaw * 180 / np.pi)
        break

needle_id = p.loadURDF('/Users/kantaphat/Research/DEX/SurRoL/surrol/assets/needle/needle_40mm.urdf',
                    needle_pos,
                    p.getQuaternionFromEuler((0, 0, yaw)),
                    useFixedBase=False,
                    globalScaling=SCALING)
p.changeVisualShape(needle_id, -1, specularColor=(80, 80, 80))

0.022024266183894886 -12.72328608636022


In [27]:
from surrol.utils.pybullet_utils import (
    get_link_pose
)

sphere_radius = 0.01
sphere_pos = get_link_pose(needle_id, 1)[0]

sphere_id = p.createMultiBody(
    baseMass=0,
    baseVisualShapeIndex=p.createVisualShape(
        p.GEOM_SPHERE, 
        radius=sphere_radius, 
        rgbaColor=[1, 1, 0, 1]
    ),
    basePosition=sphere_pos,
    baseOrientation=p.getQuaternionFromEuler([0, 0, 0])
)

In [26]:
p.removeBody(sphere_id)

b3Printf: b3Warning[examples/SharedMemory/PhysicsClientSharedMemory.cpp,1343]:

b3Printf: Removing body failed


In [13]:
get_link_pose(needle_id, 6)

[[2.757152795791626, -0.34754452109336853, 3.4860000610351562],
 [0.0, 0.0, 0.6244025826454163, 0.78110271692276]]

In [12]:
get_link_pose(needle_id, 7)

[[2.801201343536377, -0.15245549380779266, 3.4860000610351562],
 [0.0, 0.0, -0.7811025381088257, 0.6244027614593506]]

In [59]:
p.removeBody(needle_id)

In [52]:
count = 0
for _ in range(100):
    yaw = (np.random.rand() - 0.5) * np.pi
    needle_pos = (
        np.random.uniform(needle_ranges[0][0], needle_ranges[0][1]),
        workspace_limits[1][0],
        workspace_limits[2][0] + 0.01
    )

    pick_up_pos = (
        needle_pos[0] - needle_radius * np.cos(yaw),
        needle_pos[1] - needle_radius * np.sin(yaw),
        needle_pos[2]
    )

    x, y, _ = pick_up_pos
    x_min, x_max = pick_up_ranges[0][0], pick_up_ranges[0][1]
    y_min, y_max = pick_up_ranges[1][0], pick_up_ranges[1][1]
    
    if (x_min <= x <= x_max and y_min <= y <= y_max):
        print(pick_up_pos[0] - pick_up_ranges[0].mean(), 0.25 + pick_up_pos[1])

        if 0.25 + pick_up_pos[1] < 0.05:
            count += 1

print("count:", count)

-0.0009458318062440796 0.06834929776982224
-0.022352474148394474 0.0187379711892548
-0.03456454374032525 0.041731036880810896
-0.017856744631371768 0.07380631053141706
-0.044301376475973875 0.05852216213490907
0.02967682496095847 0.07506234757862024
0.008639495829859012 0.09999645430787757
0.0643037988438655 0.09985711106550305
0.04431710247424281 0.09999934935322624
-0.00286098320636885 0.06681145909154895
0.005175667810843887 0.07292710069931524
-0.00764418642614606 0.05481982391401982
0.02728248912353992 0.09744034117137973
-0.008978783302055593 0.09426603745786616
-0.09059226983353197 0.018275305295829353
0.012734517872358797 0.0998003296806704
0.04566129379903616 0.09409524633281979
-0.01619549306949697 0.09740715983188256
0.03265646509826059 0.09959432219441305
0.023443585532159084 0.09979069730698031
-0.07965185575370182 0.018552998755211658
-0.018935777328807468 0.07495822883038583
-0.013900686325016842 0.0943888966718302
-0.05915831768670765 0.07006364565659895
0.0865305656564

In [21]:
while True:
    p.stepSimulation()
    time.sleep(1.0/240.0)

KeyboardInterrupt: 